# CogMem Cognitive Patches — Minimal Experiment

**Goal:** Verify the cognitive patches architecture works.

**Plan:**
1. Load base model (4-bit, ~2GB VRAM)
2. Process first 100 tasks with N=4 candidates per task
3. Create patches from pass/fail contrasts (~20 patches expected)
4. Evaluate patches on remaining 1040 UNSEEN tasks
5. Compare: patched eval > cold eval = architecture works

**Hardware:** A4000 16GB. Model loaded in 4-bit via transformers (not Ollama).
Generation happens through model.generate() directly.


In [3]:
# Cell 1: Install deps + check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
!pip install transformers accelerate bitsandbytes peft datasets sentence-transformers -q
print("Deps installed")


Quadro RTX 5000, 16384 MiB, 16110 MiB
Deps installed


In [4]:
!pip install "transformers==4.43.4" "sentence-transformers==2.7.0" "huggingface-hub==0.25.0" "accelerate==0.33.0" "peft==0.13.2" "bitsandbytes==0.43.3" -q


In [ ]:
# Cell 2: Clone CogMem + load tasks
REPO_BRANCH = "feat/episode-cluster-memories"
!if [ ! -d /notebooks/CogMem/.git ]; then \
    git clone --branch {REPO_BRANCH} --single-branch https://github.com/tungooxx/CogMem.git /notebooks/CogMem; \
else \
    cd /notebooks/CogMem && git fetch origin && git checkout {REPO_BRANCH} && git pull --ff-only origin {REPO_BRANCH}; \
fi
!cd /notebooks/CogMem && pip install -e . --no-deps -q
!cd /notebooks/CogMem && git branch --show-current && git rev-parse --short HEAD

import sys
if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

import json
from pathlib import Path
from datasets import load_dataset

# Load BigCodeBench full (1140 tasks)
TASKS_PATH = "/notebooks/bigcodebench_tasks.jsonl"
if not Path(TASKS_PATH).exists():
    ds = load_dataset("bigcode/bigcodebench", split="v0.1.4")
    tasks = []
    for item in ds:
        tasks.append({
            "task_id": item["task_id"],
            "instruct_prompt": item.get("instruct_prompt", ""),
            "complete_prompt": item.get("complete_prompt", ""),
            "test": item.get("test", ""),
            "entry_point": item.get("entry_point", ""),
        })
    with open(TASKS_PATH, "w") as f:
        for t in tasks:
            f.write(json.dumps(t) + chr(10))
else:
    tasks = []
    with open(TASKS_PATH) as f:
        for line in f:
            if line.strip():
                tasks.append(json.loads(line))

print("Tasks:", len(tasks))
# Split: first 100 for patch creation, rest for evaluation
TRAIN_TASKS = tasks[:100]
EVAL_TASKS = tasks[100:]
print("Train (create patches):", len(TRAIN_TASKS))
print("Eval (test patches):", len(EVAL_TASKS))


Already on 'feat/episode-cluster-memories'
Your branch is up to date with 'origin/feat/episode-cluster-memories'.
From https://github.com/tungooxx/CogMem
 * branch            feat/episode-cluster-memories -> FETCH_HEAD
Already up to date.
feat/episode-cluster-memories
328b8aa
Tasks: 1140
Train (create patches): 200
Eval (test patches): 940


In [6]:
import inspect
import cogmem.patches.memory_bank as mb

print("loaded from:", mb.__file__)
print("has transfer selector:", "_select_transfer_episodes" in inspect.getsource(mb._distill_and_score_memory))
print("contains filtered token:", "task_func" in inspect.getsource(mb))
print(inspect.getsource(mb._extract_structural_markers))


loaded from: /notebooks/CogMem/cogmem/patches/memory_bank.py
has transfer selector: True
contains filtered token: True
def _extract_structural_markers(
    prompts: list[str],
    negative_prompts: list[str] | None = None,
    max_markers: int = DEFAULT_RETRIEVE_MARKERS,
) -> list[str]:
    positive_counts: dict[str, int] = {}
    for prompt in prompts:
        for token in set(_prompt_feature_tokens(prompt)):
            positive_counts[token] = positive_counts.get(token, 0) + 1
    negative_counts: dict[str, int] = {}
    for prompt in negative_prompts or []:
        for token in set(_prompt_feature_tokens(prompt)):
            negative_counts[token] = negative_counts.get(token, 0) + 1
    positive_total = max(len(prompts), 1)
    negative_total = max(len(negative_prompts or []), 1)
    ranked: list[tuple[float, int, float, str]] = []
    for token, positive_count in positive_counts.items():
        positive_rate = positive_count / positive_total
        negative_rate = negative_coun

In [7]:
# Cell 3: Load base model (4-bit) + embedder
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Loading model (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading embedder...")
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

free = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Model loaded. Free VRAM: {free:.1f} GB")
print("Ready for patch creation.")


Loading model (4-bit)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Cell 4: Record episodes from first 100 tasks
import time
from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution
from cogmem.patches.memory_bank import ClusterMemoryBank
from cogmem.patches.wake import generate_many_with_model, generate_with_model, find_best_contrast_pair

N_CANDIDATES = 4
MEMORY_DIR = "/notebooks/cogmem_cluster_memories"
PROGRESS_PATH = Path(MEMORY_DIR) / "episode_recording_progress.json"
RESET_CELL4_PROGRESS = False
memory_bank = ClusterMemoryBank(MEMORY_DIR)
memory_bank.load()

from peft import prepare_model_for_kbit_training
base_model = prepare_model_for_kbit_training(base_model)
print('Base model prepared for training')

train_signature = {
    "count": len(TRAIN_TASKS),
    "first_task_id": TRAIN_TASKS[0]["task_id"] if TRAIN_TASKS else "",
    "last_task_id": TRAIN_TASKS[-1]["task_id"] if TRAIN_TASKS else "",
}
if RESET_CELL4_PROGRESS and PROGRESS_PATH.exists():
    PROGRESS_PATH.unlink()

resume_state = {}
if PROGRESS_PATH.exists():
    with open(PROGRESS_PATH) as f:
        resume_state = json.load(f)

same_train_plan = resume_state.get("train_signature") == train_signature
start_idx = int(resume_state.get("next_index", 0)) if same_train_plan else 0
total_passed = int(resume_state.get("total_passed", 0)) if same_train_plan else 0
episodes_before = int(resume_state.get("episodes_before", len(memory_bank.episodes))) if same_train_plan else len(memory_bank.episodes)
start_time = time.time()

print('Resume progress file:', PROGRESS_PATH)
if start_idx > 0:
    print('Resuming Cell 4 from task {} of {} (next task index {}).'.format(start_idx + 1, len(TRAIN_TASKS), start_idx))
else:
    print('Starting Cell 4 from task 1 of {}.'.format(len(TRAIN_TASKS)))

if start_idx >= len(TRAIN_TASKS):
    print('Cell 4 already completed for the current TRAIN_TASKS selection. Set RESET_CELL4_PROGRESS=True to rerun from scratch.')

for i in range(start_idx, len(TRAIN_TASKS)):
    task = TRAIN_TASKS[i]
    task_id = task["task_id"]
    prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
    task_embedding = embedder.encode(prompt).tolist()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    candidates = []
    try:
        responses = generate_many_with_model(
            base_model,
            tokenizer,
            messages,
            n_candidates=N_CANDIDATES,
            temperature=0.8,
        )
        for response in responses:
            code = extract_code(response)
            if code and len(code.strip()) > 20:
                result = evaluate_solution(task, code, timeout=30, mode="subprocess")
                candidates.append({"code": code, "passed": result["passed"]})
    except Exception as e:
        if i < 3:
            print('  Gen error:', type(e).__name__, str(e)[:80])

    passes = [c for c in candidates if c["passed"]]
    fails = [c for c in candidates if not c["passed"]]

    if passes:
        total_passed += 1

    if passes and fails:
        best_pair, best_sim = find_best_contrast_pair(passes, fails)
        if best_pair:
            memory_bank.record_episode(
                task_id=task_id,
                prompt=prompt,
                task_embedding=task_embedding,
                failed_code=best_pair["fail"]["code"],
                passed_code=best_pair["pass"]["code"],
                pass_fail_similarity=best_sim,
            )

    memory_bank.save()
    progress_state = {
        "status": "running",
        "train_signature": train_signature,
        "next_index": i + 1,
        "last_task_id": task_id,
        "total_passed": total_passed,
        "episodes_before": episodes_before,
        "episodes_now": len(memory_bank.episodes),
        "updated_at": time.time(),
    }
    with open(PROGRESS_PATH, "w") as f:
        json.dump(progress_state, f, indent=2)

    if (i + 1) % 10 == 0 or i < 5:
        elapsed = time.time() - start_time
        processed_this_run = max(i + 1 - start_idx, 1)
        rate = processed_this_run / elapsed * 3600 if elapsed > 0 else 0
        print("[{}/{}] {}: {}P/{}F | episodes={} | pass_rate={}/{} | {:.0f}/hr".format(
            i + 1, len(TRAIN_TASKS), task_id,
            len(passes), len(fails), len(memory_bank.episodes),
            total_passed, i + 1, rate))

memory_bank.save()
with open(PROGRESS_PATH, "w") as f:
    json.dump({
        "status": "complete",
        "train_signature": train_signature,
        "next_index": len(TRAIN_TASKS),
        "last_task_id": TRAIN_TASKS[-1]["task_id"] if TRAIN_TASKS else "",
        "total_passed": total_passed,
        "episodes_before": episodes_before,
        "episodes_now": len(memory_bank.episodes),
        "updated_at": time.time(),
    }, f, indent=2)
elapsed = (time.time() - start_time) / 60
print()
print("=" * 50)
print("EPISODE RECORDING COMPLETE")
print("Tasks processed:", len(TRAIN_TASKS))
print("Tasks with passes:", total_passed)
print("New episodes recorded:", len(memory_bank.episodes) - episodes_before)
print("Time:", round(elapsed, 1), "min")
print("Memory bank stats:", memory_bank.stats())


Base model prepared for training
[1/200] BigCodeBench/0: 0P/4F | episodes=0 | pass_rate=0/1 | 99/hr
[2/200] BigCodeBench/1: 4P/0F | episodes=0 | pass_rate=1/2 | 112/hr
[3/200] BigCodeBench/2: 0P/4F | episodes=0 | pass_rate=1/3 | 113/hr
[4/200] BigCodeBench/3: 2P/2F | episodes=1 | pass_rate=2/4 | 112/hr
[5/200] BigCodeBench/4: 2P/2F | episodes=2 | pass_rate=3/5 | 119/hr
[10/200] BigCodeBench/9: 4P/0F | episodes=3 | pass_rate=6/10 | 104/hr
[20/200] BigCodeBench/19: 0P/4F | episodes=5 | pass_rate=8/20 | 79/hr
[30/200] BigCodeBench/29: 1P/3F | episodes=9 | pass_rate=15/30 | 85/hr
[40/200] BigCodeBench/39: 0P/4F | episodes=11 | pass_rate=18/40 | 83/hr
[50/200] BigCodeBench/49: 0P/4F | episodes=13 | pass_rate=20/50 | 81/hr
[60/200] BigCodeBench/59: 0P/4F | episodes=17 | pass_rate=25/60 | 82/hr
[70/200] BigCodeBench/69: 3P/1F | episodes=21 | pass_rate=29/70 | 76/hr
[80/200] BigCodeBench/79: 0P/4F | episodes=22 | pass_rate=30/80 | 75/hr
[90/200] BigCodeBench/89: 0P/4F | episodes=24 | pass_rate

In [ ]:
# Cell 4b: Build cluster memories and inspect distilled artifacts
import numpy as np
import torch
from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT
from cogmem.patches.compose import PatchedModel
from cogmem.patches.memory_bank import (
    ClusterMemoryBank,
    compute_applicability,
    score_memory_final_use,
    score_memory_promotion,
    score_memory_use,
)
from cogmem.patches.wake import generate_with_model

MEMORY_DIR = "/notebooks/cogmem_cluster_memories"
memory_bank = ClusterMemoryBank(MEMORY_DIR)
memory_bank.load()
print('Loaded episodes:', len(memory_bank.episodes))

if len(memory_bank.episodes) == 0:
    raise RuntimeError(
        "No saved episodes found in /notebooks/cogmem_cluster_memories. "
        "Run Cell 4 to completion before building cluster memories."
    )

CLUSTER_SIMILARITY_THRESHOLD = 0.62
CLUSTER_MIN_SUPPORT = 3
CLUSTER_CONTROL_EPISODES = 6

print('Cluster settings:', {
    'similarity_threshold': CLUSTER_SIMILARITY_THRESHOLD,
    'min_support': CLUSTER_MIN_SUPPORT,
    'control_episodes': CLUSTER_CONTROL_EPISODES,
})

build_stats = memory_bank.build_memories(
    base_model,
    tokenizer,
    similarity_threshold=CLUSTER_SIMILARITY_THRESHOLD,
    min_support=CLUSTER_MIN_SUPPORT,
    control_episodes=CLUSTER_CONTROL_EPISODES,
)
retrievable = [m for m in memory_bank.memories if m.retrievable]
max_support = max((m.support_count for m in memory_bank.memories), default=0)
max_reuse = max((m.reuse_count for m in memory_bank.memories), default=0)
print('Memory bank stats:', build_stats)
print('Retrievable memories:', len(retrievable))

print('=== CLUSTER MEMORY SUMMARY ===')
for memory in memory_bank.memories[:10]:
    payload = memory.retrievable_payload()
    print('Memory:', memory.memory_id)
    print('  family:', memory.family_label,
          'support:', memory.support_count,
          'promote:', round(memory.promotion_score, 3),
          'threshold(app):', round(memory.retrieval_threshold, 3),
          'retrievable:', memory.retrievable)
    print('  local_gain:', round(memory.local_support_gain, 4),
          'heldout_gain:', round(memory.held_out_steering_gain, 4),
          'transfer_gain:', round(memory.transfer_gain, 4),
          'transfer_online_gain:', round(memory.transfer_online_gain, 4),
          'transfer_rate:', round(memory.transfer_rate, 3))
    print('  recent_success:', round(memory.recent_success_rate, 3),
          'online_hurt:', round(memory.online_hurt_rate, 3),
          'utility_regression:', round(memory.utility_regression, 4),
          'redundancy:', round(memory.redundancy_penalty, 4))
    print('  neg_penalty:', round(memory.negative_steering_penalty, 4),
          'negatives:', len(memory.negative_episode_ids),
          'markers:', memory.structural_markers[:5])
    print('  payload keys:', list(payload.keys()))
    print('  patches:', payload['patch_ids'])

if not retrievable:
    print('No retrievable memories yet. Add more episodes or inspect family clustering.')
else:
    print('\n[1] Distilled artifact magnitudes:')
    for memory in retrievable[:3]:
        active = memory_bank.load_patches_for_memories([memory])
        if not active:
            print('  {}: no artifact patch loaded'.format(memory.memory_id))
            continue
        patch = active[0]
        norms = []
        for _, w in patch.lora_weights.items():
            nA = torch.norm(w['A']).item()
            nB = torch.norm(w['B']).item()
            norms.append(nA + nB)
        avg = sum(norms) / len(norms) if norms else 0.0
        first_key = list(patch.lora_weights.keys())[0]
        w = patch.lora_weights[first_key]
        print('  {} -> {}: |A|={:.4f} |B|={:.4f} avg_norm={:.4f}'.format(
            memory.memory_id[:36], patch.patch_id[:36],
            torch.norm(w['A']).item(), torch.norm(w['B']).item(), avg))
        patch.unload_weights()

    print('\n[2] Output difference (first 3 eval tasks):')
    for task in EVAL_TASKS[:3]:
        prompt = task.get('instruct_prompt', task.get('complete_prompt', ''))
        emb = embedder.encode(prompt).tolist()
        emb_arr = np.asarray(emb, dtype=np.float32)
        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]

        torch.manual_seed(42)
        out_cold = generate_with_model(base_model, tokenizer, messages, temperature=0)

        active_memories, active = memory_bank.get_active_patches(emb, prompt, top_k=5, return_memories=True)

        torch.manual_seed(42)
        try:
            with PatchedModel(base_model, active, scaling_factor=0.25):
                out_patched = generate_with_model(base_model, tokenizer, messages, temperature=0)
        finally:
            for patch in active:
                patch.unload_weights()

        if not active_memories:
            print('  {}: ABSTAINED'.format(task['task_id']))
            continue

        best = active_memories[0]
        applicability = compute_applicability(best, emb_arr, prompt)
        use_score = score_memory_use(best, emb_arr, prompt, max_reuse=max_reuse)
        final_use = score_memory_final_use(best, emb_arr, prompt, max_reuse=max_reuse)
        promote = score_memory_promotion(best, max_support=max_support)
        if out_cold == out_patched:
            print('  {}: IDENTICAL | final_use={:.3f} use={:.3f} app={:.3f} promote={:.3f}'.format(
                task['task_id'], final_use, use_score, applicability, promote))
        else:
            cold_tokens = out_cold.split()
            patched_tokens = out_patched.split()
            diff = sum(1 for a, b in zip(cold_tokens, patched_tokens) if a != b)
            total = max(len(cold_tokens), len(patched_tokens), 1)
            print('  {}: {:.0f}% tokens different | memory={} | final_use={:.3f} | use={:.3f} | app={:.3f} | threshold={:.3f} | promote={:.3f}'.format(
                task['task_id'], diff / total * 100, best.memory_id,
                final_use, use_score, applicability, best.retrieval_threshold, promote))



Loaded 0 patches (0 promoted) from /notebooks/cogmem_cluster_memories/patch_artifacts
Loaded episodes: 52


max_steps is given, it will override any value given in num_train_epochs
max_steps is given, it will override any value given in num_train_epochs
max_steps is given, it will override any value given in num_train_epochs
max_steps is given, it will override any value given in num_train_epochs
max_steps is given, it will override any value given in num_train_epochs
max_steps is given, it will override any value given in num_train_epochs
max_steps is given, it will override any value given in num_train_epochs


Memory bank stats: {'episodes': 52, 'memories': 7, 'retrievable_memories': 7, 'artifact_patches': 7, 'mean_promotion': 0.6725639935263213, 'mean_q': 0.6725639935263213, 'families': {'random_numeric': 1, 'networking': 1, 'file_io': 1, 'dataframe': 3, 'plotting': 1}}
Retrievable memories: 7
=== CLUSTER MEMORY SUMMARY ===
Memory: memory_random_numeric_0845eb44ce
  family: random_numeric support: 6 promote: 0.719 threshold(app): 0.365 retrievable: True
  local_gain: 1.5386 heldout_gain: 1.2594 transfer_gain: 1.2594 transfer_online_gain: 0.0 transfer_rate: 1.0
  recent_success: 0.0 online_hurt: 0.0 utility_regression: 0.0 redundancy: 0.0
  neg_penalty: 0.0 negatives: 3 markers: ['contained', 'import', 'random', 'starting', 'task_func']
  payload keys: ['cluster_metadata', 'evidence', 'transfer_stats', 'patch_ids']
  patches: ['cluster_patch_memory_random_numeric_0845eb44ce']
Memory: memory_networking_84e1653036
  family: networking support: 3 promote: 0.702 threshold(app): 0.358 retrievable

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  BigCodeBench/200: 74% tokens different | memory=memory_plotting_461f4c79e9 | final_use=0.373 | use=0.262 | app=0.403 | threshold=0.381 | promote=0.740
  BigCodeBench/201: ABSTAINED
  BigCodeBench/202: ABSTAINED


In [ ]:
from cogmem.patches.memory_bank import ClusterMemoryBank
memory_bank = ClusterMemoryBank("/notebooks/cogmem_cluster_memories")
memory_bank.load()
print(f"Episodes: {len(memory_bank.episodes)}")
print(f"Memories: {len(memory_bank.memories)}")
print(f"Artifact patches: {len(memory_bank.artifact_bank.patches)}")


2026-04-16 09:31:52.026136: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-16 09:31:52.026213: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-16 09:31:52.027836: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-16 09:31:52.035744: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-16 09:31:53.248988: W tensorflow/compiler/tf2

Loaded 7 patches (0 promoted) from /notebooks/cogmem_cluster_memories/patch_artifacts
Episodes: 52
Memories: 7
Artifact patches: 7


In [ ]:
# Cell 4c: Check composition works with retrieved cluster memories
import numpy as np
import torch
from cogmem.patches.compose import PatchedModel
from cogmem.patches.memory_bank import (
    compute_applicability,
    score_memory_final_use,
    score_memory_promotion,
    score_memory_use,
)

task = EVAL_TASKS[0]
prompt = task.get('instruct_prompt', task.get('complete_prompt', ''))
emb = embedder.encode(prompt).tolist()
emb_arr = np.asarray(emb, dtype=np.float32)
messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]
max_support = max((m.support_count for m in memory_bank.memories), default=0)
max_reuse = max((m.reuse_count for m in memory_bank.memories), default=0)

active_memories, active = memory_bank.get_active_patches(emb, prompt, top_k=5, return_memories=True)
print('Selected memories:', [m.memory_id for m in active_memories])
print('Artifact patches:', [p.patch_id for p in active])
if active_memories:
    best = active_memories[0]
    print('Applicability:', round(compute_applicability(best, emb_arr, prompt), 4))
    print('Q_use      :', round(score_memory_use(best, emb_arr, prompt, max_reuse=max_reuse), 4))
    print('FinalUse   :', round(score_memory_final_use(best, emb_arr, prompt, max_reuse=max_reuse), 4))
    print('Q_promote  :', round(score_memory_promotion(best, max_support=max_support), 4))
    print('Threshold  :', round(best.retrieval_threshold, 4))
    print('Payload keys:', list(best.retrievable_payload().keys()))
else:
    print('ABSTAINED: no memory cleared threshold')

print()
print('=== Greedy (temp=0) ===')
torch.manual_seed(42)
out_cold = generate_with_model(base_model, tokenizer, messages, temperature=0)

torch.manual_seed(42)
with PatchedModel(base_model, active, scaling_factor=0.25):
    out_patched = generate_with_model(base_model, tokenizer, messages, temperature=0)

print('Cold first 100:', out_cold[:100])
print('Patched first 100:', out_patched[:100])
print('IDENTICAL:', out_cold == out_patched)

print()
print('=== Hook Count ===')
pm = PatchedModel(base_model, active, scaling_factor=0.25)
pm.__enter__()
print('Hooks registered:', len(pm._hooks))
pm.__exit__(None, None, None)
print('Hooks removed:', len(pm._hooks) == 0)

print()
print('=== Low-temp (0.01) ===')
torch.manual_seed(42)
out_cold_lt = generate_with_model(base_model, tokenizer, messages, temperature=0.01)

torch.manual_seed(42)
with PatchedModel(base_model, active, scaling_factor=0.25):
    out_patched_lt = generate_with_model(base_model, tokenizer, messages, temperature=0.01)

print('IDENTICAL:', out_cold_lt == out_patched_lt)

for patch in active:
    patch.unload_weights()



/usr/local/lib/python3.11/dist-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Selected memories: ['memory_plotting_461f4c79e9']
Artifact patches: ['cluster_patch_memory_plotting_461f4c79e9']
Applicability: 0.4033
Q_use      : 0.2622
FinalUse   : 0.3732
Q_promote  : 0.74
Threshold  : 0.3814
Payload keys: ['cluster_metadata', 'evidence', 'transfer_stats', 'patch_ids']

=== Greedy (temp=0) ===
Cold first 100: Thought: 
To solve this task, I need to generate 'n' random numbers between 0 and 1. Then, I'll calc
Patched first 100: import random
import bisect
import statistics
import matplotlib.pyplot as plt

def task_func(n, valu
IDENTICAL: False

=== Hook Count ===
Hooks registered: 144
Hooks removed: True

=== Low-temp (0.01) ===
IDENTICAL: False


In [ ]:
# Cell 5: Inspect applicability + final use on the first 200 unseen tasks
import numpy as np
from collections import Counter
from cogmem.patches.memory_bank import compute_applicability, score_memory_final_use, score_memory_use

UNSEEN_SIM_SIZE = 200
unseen_subset = EVAL_TASKS[:UNSEEN_SIM_SIZE]
retrievable_memories = [m for m in memory_bank.memories if m.retrievable]
max_reuse = max((m.reuse_count for m in memory_bank.memories), default=0)
print("Unseen tasks inspected:", len(unseen_subset))
print("Retrievable memories:", len(retrievable_memories))

if not retrievable_memories:
    print("No retrievable memories yet.")
else:
    top1_final_use = []
    top1_use = []
    top1_applicability = []
    margins = []
    abstained = 0
    memory_hits = Counter()
    family_hits = Counter()
    rows = []

    for task in unseen_subset:
        prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
        emb = np.asarray(embedder.encode(prompt).tolist(), dtype=np.float32)
        scored = []
        for memory in retrievable_memories:
            applicability = compute_applicability(memory, emb, prompt)
            if applicability <= memory.retrieval_threshold:
                continue
            use_score = score_memory_use(memory, emb, prompt, max_reuse=max_reuse)
            final_use = score_memory_final_use(memory, emb, prompt, max_reuse=max_reuse)
            scored.append((final_use, use_score, applicability, memory))

        if not scored:
            abstained += 1
            rows.append((task['task_id'], 0.0, 0.0, 0.0, 'ABSTAIN', '', False))
            continue

        scored.sort(key=lambda item: item[0], reverse=True)
        best_final_use, best_use, best_applicability, best_memory = scored[0]
        top1_final_use.append(best_final_use)
        top1_use.append(best_use)
        top1_applicability.append(best_applicability)
        margins.append(best_applicability - best_memory.retrieval_threshold)
        memory_hits[best_memory.memory_id] += 1
        family_hits[best_memory.family_label] += 1
        rows.append((
            task['task_id'], best_final_use, best_use, best_applicability,
            best_memory.memory_id, best_memory.family_label, True,
        ))

    print("Mean top-1 final use: {:.3f}".format(float(np.mean(top1_final_use)) if top1_final_use else 0.0))
    print("Mean top-1 Q_use: {:.3f}".format(float(np.mean(top1_use)) if top1_use else 0.0))
    print("Mean top-1 applicability: {:.3f}".format(float(np.mean(top1_applicability)) if top1_applicability else 0.0))
    print("Mean applicability margin: {:.3f}".format(float(np.mean(margins)) if margins else 0.0))
    print("Abstentions: {}/{} ({:.1%})".format(abstained, len(unseen_subset), abstained / max(len(unseen_subset), 1)))

    print()
    print("Most selected memories (after gate):")
    for memory_id, hits in memory_hits.most_common():
        print("  {} -> {} tasks".format(memory_id, hits))

    print()
    print("Most selected families (after gate):")
    for family, hits in family_hits.most_common():
        print("  {} -> {} tasks".format(family, hits))

    print()
    print("Top unseen examples:")
    for task_id, final_use, use_score, applicability, memory_id, family, passed_gate in rows[:10]:
        print("  {} | final_use={:.3f} | use={:.3f} | app={:.3f} | memory={} | family={} | selected={}".format(
            task_id, final_use, use_score, applicability, memory_id, family, passed_gate
        ))



Unseen tasks inspected: 200
Retrievable memories: 7
Mean top-1 final use: 0.365
Mean top-1 Q_use: 0.256
Mean top-1 applicability: 0.394
Mean applicability margin: 0.037
Abstentions: 107/200 (53.5%)

Most selected memories (after gate):
  memory_file_io_040cb3229f -> 37 tasks
  memory_plotting_461f4c79e9 -> 35 tasks
  memory_random_numeric_0845eb44ce -> 12 tasks
  memory_dataframe_2b3ba718c6 -> 7 tasks
  memory_dataframe_d144959b3a -> 1 tasks
  memory_networking_84e1653036 -> 1 tasks

Most selected families (after gate):
  file_io -> 37 tasks
  plotting -> 35 tasks
  random_numeric -> 12 tasks
  dataframe -> 8 tasks
  networking -> 1 tasks

Top unseen examples:
  BigCodeBench/200 | final_use=0.373 | use=0.262 | app=0.403 | memory=memory_plotting_461f4c79e9 | family=plotting | selected=True
  BigCodeBench/201 | final_use=0.000 | use=0.000 | app=0.000 | memory=ABSTAIN | family= | selected=False
  BigCodeBench/202 | final_use=0.000 | use=0.000 | app=0.000 | memory=ABSTAIN | family= | selec

In [ ]:
# Cell 5b: Sweep gated retrieval width on a small seen slice
import numpy as np
from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution
from cogmem.patches.compose import PatchedModel
from cogmem.patches.wake import generate_with_model

DIAG_SIZE = 30
TOPK_OPTIONS = [1, 2, 5]
SCALE = 0.25

seen_subset = TRAIN_TASKS[:DIAG_SIZE]
print('Running gated retrieval sweep on', len(seen_subset), 'seen tasks')

cached = []
for i, task in enumerate(seen_subset):
    prompt = task.get('instruct_prompt', task.get('complete_prompt', ''))
    emb = embedder.encode(prompt).tolist()
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]

    cold_ok = False
    try:
        cold_response = generate_with_model(base_model, tokenizer, messages, temperature=0)
        cold_code = extract_code(cold_response)
        cold_result = evaluate_solution(task, cold_code, timeout=30, mode='subprocess')
        cold_ok = bool(cold_result['passed'])
    except Exception:
        cold_ok = False

    cached.append({
        'task': task,
        'prompt': prompt,
        'embedding': emb,
        'messages': messages,
        'cold_ok': cold_ok,
    })

    if (i + 1) % 10 == 0 or i + 1 == len(seen_subset):
        cold_passed = sum(1 for row in cached if row['cold_ok'])
        print(f'  cached cold [{i+1}/{len(seen_subset)}] cold={cold_passed}', flush=True)

results = []
cold_passed = sum(1 for row in cached if row['cold_ok'])

for top_k in TOPK_OPTIONS:
    print()
    print(f'-- top_k={top_k}, scale={SCALE} --', flush=True)
    memory_passed = 0
    helped = 0
    hurt = 0
    abstained = 0

    for i, row in enumerate(cached):
        mem_ok = row['cold_ok']
        active_memories = []
        active_patches = []

        try:
            active_memories, active_patches = memory_bank.get_active_patches(
                row['embedding'], row['prompt'], top_k=top_k, return_memories=True
            )
            if not active_patches:
                abstained += 1
            else:
                with PatchedModel(base_model, active_patches, scaling_factor=SCALE):
                    mem_response = generate_with_model(base_model, tokenizer, row['messages'], temperature=0)
                mem_code = extract_code(mem_response)
                mem_result = evaluate_solution(row['task'], mem_code, timeout=30, mode='subprocess')
                mem_ok = bool(mem_result['passed'])
        except Exception:
            mem_ok = False
        finally:
            for patch in active_patches:
                patch.unload_weights()

        if mem_ok:
            memory_passed += 1
        if (not row['cold_ok']) and mem_ok:
            helped += 1
        elif row['cold_ok'] and (not mem_ok):
            hurt += 1

        if (i + 1) % 10 == 0 or i + 1 == len(cached):
            print(f'  [{i+1}/{len(cached)}] memory={memory_passed} helped={helped} hurt={hurt} abstain={abstained}', flush=True)

    results.append({
        'top_k': top_k,
        'cold_passed': cold_passed,
        'memory_passed': memory_passed,
        'cold_rate': cold_passed / len(cached),
        'memory_rate': memory_passed / len(cached),
        'delta': (memory_passed - cold_passed) / len(cached),
        'helped': helped,
        'hurt': hurt,
        'abstained': abstained,
    })

results.sort(key=lambda r: (r['delta'], r['memory_rate'], -r['hurt']), reverse=True)
print()
print('{:<5} {:>8} {:>8} {:>8} {:>8} {:>8}'.format('k', 'cold', 'memory', 'delta', 'hurt', 'abstain'))
print('-' * 64)
for r in results:
    print('{:<5} {:>7.1%} {:>7.1%} {:>+7.1%} {:>8} {:>8}'.format(
        r['top_k'], r['cold_rate'], r['memory_rate'], r['delta'], r['hurt'], r['abstained']))
print()
print('Best setting:', results[0])


Running gated retrieval sweep on 30 seen tasks


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  cached cold [10/30] cold=4
  cached cold [20/30] cold=4
  cached cold [30/30] cold=8

-- top_k=1, scale=0.25 --


/usr/local/lib/python3.11/dist-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


  [10/30] memory=5 helped=1 hurt=0 abstain=7
  [20/30] memory=8 helped=4 hurt=0 abstain=11
  [30/30] memory=12 helped=4 hurt=0 abstain=21

-- top_k=2, scale=0.25 --
  [10/30] memory=5 helped=1 hurt=0 abstain=7
  [20/30] memory=8 helped=4 hurt=0 abstain=11
  [30/30] memory=12 helped=4 hurt=0 abstain=21

-- top_k=5, scale=0.25 --
  [10/30] memory=5 helped=1 hurt=0 abstain=7
  [20/30] memory=8 helped=4 hurt=0 abstain=11
  [30/30] memory=12 helped=4 hurt=0 abstain=21

k         cold   memory    delta     hurt  abstain
----------------------------------------------------------------
1       26.7%   40.0%  +13.3%        0       21
2       26.7%   40.0%  +13.3%        0       21
5       26.7%   40.0%  +13.3%        0       21

Best setting: {'top_k': 1, 'cold_passed': 8, 'memory_passed': 12, 'cold_rate': 0.26666666666666666, 'memory_rate': 0.4, 'delta': 0.13333333333333333, 'helped': 4, 'hurt': 0, 'abstained': 21}


In [ ]:
# Cell 6: Evaluate gated cluster memories on seen and unseen tasks
import json
import logging
import traceback
from pathlib import Path

from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution
from cogmem.patches.compose import PatchedModel
from cogmem.patches.wake import generate_with_model

EVAL_TOP_K = 5
EVAL_SCALE = 0.25
EVAL_CACHE_VERSION = 'finaluse_v4'
SEEN_EVAL_TASKS = TRAIN_TASKS
UNSEEN_EVAL_SIZE = 50
UNSEEN_EVAL_TASKS = EVAL_TASKS[:UNSEEN_EVAL_SIZE]
FORCE_RERUN_EVAL = False
EVAL_CACHE_PATH = Path('/notebooks/cogmem_cluster_memories') / (
    f'eval_cache_{EVAL_CACHE_VERSION}_seen{len(SEEN_EVAL_TASKS)}_unseen{UNSEEN_EVAL_SIZE}_'
    f'top{EVAL_TOP_K}_scale{str(EVAL_SCALE).replace(".", "p")}.json'
)

logger = logging.getLogger('paperspace.eval')
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter('[%(levelname)s] %(message)s'))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)

print('Seen tasks:', len(SEEN_EVAL_TASKS))
print('Unseen tasks:', len(UNSEEN_EVAL_TASKS))
print('Episodes:', len(memory_bank.episodes))
print('Memories:', len(memory_bank.memories))
print('Artifact patches:', len(memory_bank.artifact_bank.patches))
print('Eval cache:', EVAL_CACHE_PATH)

def run_eval(tasks, label):
    print()
    print('--- {} COLD + MEMORY EVAL ---'.format(label))
    cold_passed = 0
    memory_passed = 0
    abstained = 0
    used_memory = 0

    for i, task in enumerate(tasks):
        task_id = task.get('task_id', task.get('id', f'{label}_{i}'))
        prompt = task.get('instruct_prompt', task.get('complete_prompt', ''))
        task_embedding = embedder.encode(prompt).tolist()
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ]

        cold_ok = False
        mem_ok = False
        active_memories = []
        active_patches = []

        try:
            cold_response = generate_with_model(base_model, tokenizer, messages, temperature=0)
            cold_code = extract_code(cold_response)
            cold_result = evaluate_solution(task, cold_code, timeout=30, mode='subprocess')
            cold_ok = bool(cold_result['passed'])

            active_memories, active_patches = memory_bank.get_active_patches(
                task_embedding,
                prompt,
                top_k=EVAL_TOP_K,
                return_memories=True,
            )
            if not active_patches:
                abstained += 1
                mem_ok = cold_ok
            else:
                used_memory += 1
                with PatchedModel(base_model, active_patches, scaling_factor=EVAL_SCALE):
                    mem_response = generate_with_model(base_model, tokenizer, messages, temperature=0)
                mem_code = extract_code(mem_response)
                mem_result = evaluate_solution(task, mem_code, timeout=30, mode='subprocess')
                mem_ok = bool(mem_result['passed'])
        except Exception as exc:
            logger.warning(
                'Task %s failed during generate_with_model/evaluate_solution/PatchedModel: %s: %s\n%s',
                task_id,
                type(exc).__name__,
                exc,
                ''.join(traceback.format_exc(limit=3)),
            )
            mem_ok = False
        finally:
            for patch in active_patches:
                patch.unload_weights()

        if cold_ok:
            cold_passed += 1
        if mem_ok:
            memory_passed += 1

        if len(active_memories) == 1:
            memory_bank.update_memory_utility(
                active_memories[0].memory_id,
                task_succeeded=mem_ok,
                cold_succeeded=cold_ok,
                eval_split=label.lower(),
                persist=False,
            )

        if (i + 1) % 50 == 0 or i + 1 == len(tasks):
            print('  [{}/{}] cold: {}/{} ({:.1%}) | memory: {}/{} ({:.1%}) | abstain={}'.format(
                i + 1, len(tasks),
                cold_passed, i + 1, cold_passed / max(i + 1, 1),
                memory_passed, i + 1, memory_passed / max(i + 1, 1),
                abstained,
            ))

    cold_rate = cold_passed / max(len(tasks), 1)
    memory_rate = memory_passed / max(len(tasks), 1)
    print('{} cold result: {} / {} ({:.1%})'.format(label, cold_passed, len(tasks), cold_rate))
    print('{} memory result: {} / {} ({:.1%})'.format(label, memory_passed, len(tasks), memory_rate))
    print('{} memory usage: used={} abstained={} ({:.1%} abstain)'.format(
        label, used_memory, abstained, abstained / max(len(tasks), 1)
    ))
    return {
        'label': label,
        'total': len(tasks),
        'cold_passed': cold_passed,
        'memory_passed': memory_passed,
        'cold_rate': cold_rate,
        'memory_rate': memory_rate,
        'delta': memory_rate - cold_rate,
        'used_memory': used_memory,
        'abstained': abstained,
    }

if EVAL_CACHE_PATH.exists() and not FORCE_RERUN_EVAL:
    cached = json.loads(EVAL_CACHE_PATH.read_text())
    seen_eval = cached['seen_eval']
    unseen_eval = cached['unseen_eval']
    print('Loaded cached eval results from', EVAL_CACHE_PATH)
else:
    seen_eval = run_eval(SEEN_EVAL_TASKS, 'SEEN')
    unseen_eval = run_eval(UNSEEN_EVAL_TASKS, 'UNSEEN')
    EVAL_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    EVAL_CACHE_PATH.write_text(json.dumps({
        'seen_eval': seen_eval,
        'unseen_eval': unseen_eval,
    }, indent=2))
    print('Saved eval cache to', EVAL_CACHE_PATH)

memory_bank.run_sleep_cycle(prune=True)
memory_bank.save()



Seen tasks: 200
Unseen tasks: 50
Episodes: 52
Memories: 7
Artifact patches: 7
Eval cache: /notebooks/cogmem_cluster_memories/eval_cache_finaluse_v4_seen200_unseen50_top5_scale0p25.json

--- SEEN COLD + MEMORY EVAL ---
  [50/200] cold: 10/50 (20.0%) | memory: 14/50 (28.0%) | abstain=30
  [100/200] cold: 24/100 (24.0%) | memory: 25/100 (25.0%) | abstain=72
  [150/200] cold: 39/150 (26.0%) | memory: 37/150 (24.7%) | abstain=109
  [200/200] cold: 51/200 (25.5%) | memory: 50/200 (25.0%) | abstain=155
SEEN cold result: 51 / 200 (25.5%)
SEEN memory result: 50 / 200 (25.0%)
SEEN memory usage: used=45 abstained=155 (77.5% abstain)

--- UNSEEN COLD + MEMORY EVAL ---
  [50/50] cold: 9/50 (18.0%) | memory: 9/50 (18.0%) | abstain=50
UNSEEN cold result: 9 / 50 (18.0%)
UNSEEN memory result: 9 / 50 (18.0%)
UNSEEN memory usage: used=0 abstained=50 (100.0% abstain)
Saved eval cache to /notebooks/cogmem_cluster_memories/eval_cache_finaluse_v4_seen200_unseen50_top5_scale0p25.json


In [ ]:
# Cell 7: Results comparison + current score formulas
print('=' * 72)
print('EPISODE-FIRST CLUSTER MEMORY RESULTS')
print('=' * 72)
print()
print('Episodes recorded:', len(memory_bank.episodes))
print('Cluster memories built:', len(memory_bank.memories))
print('Artifact patches available:', len(memory_bank.artifact_bank.patches))
print('Eval top_k:', EVAL_TOP_K, '| Eval scale:', EVAL_SCALE)
print()
print('{:<12} {:>8} {:>8} {:>9} {:>9} {:>9} {:>10}'.format('Split', 'Cold', 'Memory', 'Cold %', 'Mem %', 'Delta', 'Abstain'))
print('-' * 72)
for result in [seen_eval, unseen_eval]:
    print('{:<12} {:>8} {:>8} {:>8.1%} {:>8.1%} {:>+8.1%} {:>9.1%}'.format(
        result['label'],
        result['cold_passed'],
        result['memory_passed'],
        result['cold_rate'],
        result['memory_rate'],
        result['delta'],
        result['abstained'] / max(result['total'], 1),
    ))

print()
print('Current wake retrieval:')
print('  applicability = clip(0.60 * pos_sim - 0.25 * neg_sim + 0.15 * structural_match - 0.10 * hard_negative_margin_penalty, 0, 1)')
print('  Q_use = applicability * clip(0.45 * transfer_gain + 0.20 * recent_success_rate + 0.15 * log_reuse - 0.20 * online_hurt_rate, 0, 1)')
print('  FinalUse = clip(Q_use + 0.15 * Q_promote, 0, 1)')
print('  drop memories with applicability <= retrieval_threshold')
print('  use top-1 unless FinalUse is very high and nearby memories stay within a small margin')
print()
print('Current sleep promotion:')
print('  Q_promote = 0.28 * heldout_gain + 0.16 * transfer_gain + 0.06 * transfer_online_gain + 0.12 * local_support_gain')
print('             + 0.10 * distillation_success + 0.08 * log_support + 0.10 * recent_success_rate')
print('             - 0.10 * online_hurt_rate - 0.15 * utility_regression - 0.15 * unseen_hurt_rate - 0.10 * redundancy_penalty')
print('  promote if Q_promote >= 0.40 and support_count >= 3')
print('  demote if preserve set is harmed')
print('  prune if Q_promote <= 0.10, support_count < 3, and preserve set is harmed')
print('  legacy q_value mirrors promotion_score for compatibility')

print()
if unseen_eval['delta'] > 0.01:
    print('Unseen-task memory improvement is positive.')
elif unseen_eval['delta'] > -0.01:
    print('Unseen-task memory effect is roughly neutral.')
else:
    print('Unseen-task memory effect is negative.')

print()
print('Memory bank:')
for key, value in memory_bank.stats().items():
    print('  {}: {}'.format(key, value))



EPISODE-FIRST CLUSTER MEMORY RESULTS

Episodes recorded: 52
Cluster memories built: 7
Artifact patches available: 7
Eval top_k: 5 | Eval scale: 0.25

Split            Cold   Memory    Cold %     Mem %     Delta    Abstain
------------------------------------------------------------------------
SEEN               51       50    25.5%    25.0%    -0.5%     77.5%
UNSEEN              9        9    18.0%    18.0%    +0.0%    100.0%

Current wake retrieval:
  applicability = clip(0.60 * pos_sim - 0.25 * neg_sim + 0.15 * structural_match - 0.10 * hard_negative_margin_penalty, 0, 1)
  Q_use = applicability * clip(0.45 * transfer_gain + 0.20 * recent_success_rate + 0.15 * log_reuse - 0.20 * online_hurt_rate, 0, 1)
  FinalUse = clip(Q_use + 0.15 * Q_promote, 0, 1)
  drop memories with applicability <= retrieval_threshold
  use top-1 unless FinalUse is very high and nearby memories stay within a small margin

Current sleep promotion:
  Q_promote = 0.28 * heldout_gain + 0.16 * transfer_gain + 0.06